# Análisis de la Balanza de Pagos - BCRP

Fuente: [BCRP - Estadísticas](https://www.bcrp.gob.pe/estadisticas.html)

Este notebook carga y prepara la data de Balanza de Pagos para su análisis y visualización.

## 1. Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.grid'] = False
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['figure.dpi'] = 200
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print('Configuración lista ✅')


In [ ]:
# Funcion para graficar tablas formato APA 7ma edicion
def plot_table(data, title='', figsize=None, note='', fname=None):
    if fname:
        os.makedirs(os.path.dirname(fname), exist_ok=True)
    if figsize is None:
        ncols = len(data.columns)
        figsize = (max(4, ncols * 1.8), max(1.5, len(data) * 0.35 + 0.8))
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    col_labels = [str(c).replace('_', ' ') for c in data.columns]
    cell_text = []
    for _, row in data.iterrows():
        cell_text.append([
            f'{v:.0f}' if isinstance(v, (int, float)) and not np.isnan(v) and v == int(v)
            else f'{v:.1f}' if isinstance(v, (int, float)) and not np.isnan(v)
            else str(v)
            for v in row
        ])
    tbl = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        cellLoc='center',
        loc='center',
        colColours=['white'] * len(data.columns),
        cellColours=[['white'] * len(data.columns) for _ in range(len(data))]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.6)
    n_rows = len(data) + 1
    n_cols = len(data.columns)
    for (row, col), cell_obj in tbl.get_celld().items():
        cell_obj.set_linewidth(0)
        cell_obj.visible_edges = ''
        cell_obj.set_text_props(fontfamily='serif', fontsize=8)
        if row == 0:
            cell_obj.set_text_props(fontfamily='serif', fontweight='bold', fontsize=8)
    for c in range(n_cols):
        tbl[0, c].visible_edges = 'TB'
        tbl[0, c].set_linewidth(1.2)
        tbl[0, c].set_edgecolor('black')
    for c in range(n_cols):
        tbl[n_rows - 1, c].visible_edges = 'B'
        tbl[n_rows - 1, c].set_linewidth(1.2)
        tbl[n_rows - 1, c].set_edgecolor('black')
    tbl.auto_set_column_width(col=list(range(n_cols)))
    if title:
        ax.set_title(title, fontfamily='serif', fontsize=10, fontweight='bold', pad=12)
    if note:
        ax.text(0.5, -0.08, note, transform=ax.transAxes,
                fontfamily='serif', fontsize=7.5, ha='center', va='top', fontstyle='italic')
    if fname:
        plt.savefig(fname, bbox_inches='tight', dpi=200)
    plt.tight_layout()
    plt.show()


## 2. Carga de datos

El CSV tiene una estructura particular:
- **Fila 1**: vacía (se saltea)
- **Fila 2**: códigos de serie (PM39985BA, etc.)
- **Fila 3**: descripciones largas de cada serie
- **Filas 4+**: datos con el año en la primera columna
- **Encoding**: ISO-8859-1 (Latin-1), **no** UTF-8

In [ ]:
# Carga cruda: saltamos fila 1 (vacía), sin cabecera
df_raw = pd.read_csv(
    'assets/Balanza_de_Pagos.csv',
    encoding='ISO-8859-1',
    skiprows=1,
    header=None
)

print(f'Dimensiones raw: {df_raw.shape}')
df_raw.head(3)

### 2.1 Separar metadatos y datos

In [ ]:
# Códigos de serie (fila 0 del raw)
codigos = df_raw.iloc[0, 1:].tolist()

# Descripciones largas (fila 1 del raw)
descripciones = df_raw.iloc[1, 1:].tolist()

# Años disponibles
años = df_raw.iloc[2:, 0].tolist()

print(f'{len(codigos)} series cargadas')
print(f'Años: {años[0]} – {años[-2]} (2026 sin data)')
print(f'Códigos de ejemplo: {codigos[:3]}...')

### 2.2 Construir DataFrame limpio

In [ ]:
# Nombres cortos para trabajar más cómodo
nombres_cortos = [
    'CF_Sector_Privado',          # 1  PM39985BA
    'Efecto_Valuacion',           # 2  PM39998BA
    'Var_Saldo_RIN',              # 3  PM39997BA
    'Resultado_BP',               # 4  PM39996BA
    'Errores_Omisiones',          # 5  PM39995BA
    'Financ_Excepcional',         # 6  PM39994BA
    'CF_CortoPlazo_Pasivos',      # 7  PM39993BA
    'CF_CortoPlazo_Activos',      # 8  PM39992BA
    'CF_CortoPlazo',              # 9  PM39991BA
    'CF_SectorPublico_Pasivos',   # 10 PM39990BA
    'CF_SectorPublico_Activos',   # 11 PM39989BA
    'CF_SectorPublico',           # 12 PM39988BA
    'CF_SectorPrivado_Pasivos',   # 13 PM39987BA
    'CF_SectorPrivado_Activos',   # 14 PM39986BA
    'Cuenta_Corriente',           # 15 PM39972BA
    'Cuenta_Financiera',          # 16 PM39984BA
    'CC_Remesas',                 # 17 PM39983BA
    'CC_Ingreso_Secundario',      # 18 PM39982BA
    'CC_Ingreso_Primario_Pub',    # 19 PM39981BA
    'CC_Ingreso_Primario_Priv',   # 20 PM39980BA
    'CC_Ingreso_Primario',        # 21 PM39979BA
    'CC_Servicios_Import',        # 22 PM39978BA
    'CC_Servicios_Export',        # 23 PM39977BA
    'CC_Servicios',               # 24 PM39976BA
    'CC_Bienes_Import',           # 25 PM39975BA
    'CC_Bienes_Export',           # 26 PM39974BA
    'CC_Bienes',                  # 27 PM39973BA
]

df = (
    df_raw.iloc[2:, 1:]              # solo datos numéricos
    .set_axis(nombres_cortos, axis=1)  # nombres cortos como columnas
    .replace('n.d.', np.nan)           # 'n.d.' → NaN
    .apply(pd.to_numeric, errors='coerce')  # todo a numérico
)
df.index = [int(a) for a in años]
df.index.name = 'Año'

# Quitamos 2026 si está totalmente vacío
df = df.dropna(how='all')

print(f'DataFrame final: {df.shape[0]} años × {df.shape[1]} series')
df.head()

## 3. Diccionario de variables

Relación entre código BCRP, nombre corto y descripción completa.

In [ ]:
diccionario = pd.DataFrame({
    'Código': codigos,
    'Nombre_Corto': nombres_cortos,
    'Descripción': descripciones
})

diccionario

## 4. Estadísticas descriptivas

In [ ]:
df.describe().round(2)

In [ ]:
# Verificar valores nulos
nulos = df.isnull().sum()
nulos[nulos > 0]

---
## 5. Visualización

A partir de acá se pueden construir los gráficos deseados.

In [ ]:
# Datos del resumen
plot_table(df[['Cuenta_Corriente', 'Cuenta_Financiera', 'Errores_Omisiones', 'Financ_Excepcional', 'Resultado_BP']].round(1).reset_index(), title='Tabla 1: Resumen de Balanza de Pagos (millones US$)', note='Elaboracion propia con datos del BCRP.', fname='output/tables/tabla_1_resumen_bp.png')


In [ ]:
# Resumen (barras apiladas + linea)
fig, ax = plt.subplots(figsize=(8, 4.5))
x = range(len(df))
ax.bar(x, df['Cuenta_Corriente'], color='white', edgecolor='black', linewidth=0.5, label='Cuenta Corriente')
ax.bar(x, df['Cuenta_Financiera'], bottom=df['Cuenta_Corriente'], color='lightgray', edgecolor='black', linewidth=0.5, label='Cuenta Financiera')
ax.bar(x, df['Errores_Omisiones'], bottom=df['Cuenta_Corriente']+df['Cuenta_Financiera'], color='darkgray', edgecolor='black', linewidth=0.5, label='Errores y Omisiones')
ax.bar(x, df['Financ_Excepcional'], bottom=df['Cuenta_Corriente']+df['Cuenta_Financiera']+df['Errores_Omisiones'], color='black', edgecolor='black', linewidth=0.5, label='Financ. Excepcional')
ax.plot(x, df['Resultado_BP'], 'o-', color='black', markerfacecolor='white', markeredgecolor='black', markersize=6, linewidth=1.5, label='Resultado de Balanza de Pagos')
ax.set_xticks(list(x))
ax.set_xticklabels(df.index)
ax.set_ylabel('Millones de US$')
ax.set_xlabel('Año')
ax.legend(frameon=False, fontsize=8, ncol=2)
ax.axhline(0, color='black', linewidth=0.3)
plt.savefig('output/graph/grafico_1_resumen_bp_apilado.png', bbox_inches='tight', dpi=200)
plt.tight_layout()
fig.text(0.5, -0.02, 'Elaboracion propia con datos del BCRP.', fontfamily='serif', fontsize=7.5, ha='center', va='top', fontstyle='italic')
plt.show()


In [ ]:
# Datos de Cuenta Corriente
plot_table(df[['CC_Bienes', 'CC_Servicios', 'CC_Ingreso_Primario', 'CC_Ingreso_Secundario', 'Cuenta_Corriente']].round(1).reset_index(), title='Tabla 2: Componentes de Cuenta Corriente (millones US$)', note='Elaboracion propia con datos del BCRP.', fname='output/tables/tabla_2_componentes_cc.png')


In [ ]:
# Datos de Bienes y Servicios
plot_table(df[['CC_Bienes_Export', 'CC_Bienes_Import', 'CC_Bienes', 'CC_Servicios_Export', 'CC_Servicios_Import', 'CC_Servicios']].round(1).reset_index(), title='Tabla 3: Desglose de Bienes y Servicios (millones US$)', note='Elaboracion propia con datos del BCRP.', fname='output/tables/tabla_3_desglose_bienes_servicios.png')


In [ ]:
# Desglose de Bienes y Servicios
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
ax1.plot(df.index, df['CC_Bienes_Export'], 'o-', color='black', markerfacecolor='white', markersize=4, label='Exportaciones')
ax1.plot(df.index, df['CC_Bienes_Import'], 's--', color='black', markerfacecolor='black', markersize=4, label='Importaciones')
ax1.plot(df.index, df['CC_Bienes'], '^-', color='black', markerfacecolor='white', markersize=4, label='Saldo neto')
ax1.axhline(0, color='black', linewidth=0.3)
ax1.set_title('Balanza Comercial', fontsize=10)
ax1.set_ylabel('Millones de US$')
ax1.set_xlabel('Año')
ax1.legend(frameon=False, fontsize=8)
ax2.plot(df.index, df['CC_Servicios_Export'], 'o-', color='black', markerfacecolor='white', markersize=4, label='Exportaciones')
ax2.plot(df.index, df['CC_Servicios_Import'], 's--', color='black', markerfacecolor='black', markersize=4, label='Importaciones')
ax2.plot(df.index, df['CC_Servicios'], '^-', color='black', markerfacecolor='white', markersize=4, label='Saldo neto')
ax2.axhline(0, color='black', linewidth=0.3)
ax2.set_title('Balanza de Servicios', fontsize=10)
ax2.set_ylabel('Millones de US$')
ax2.set_xlabel('Año')
ax2.legend(frameon=False, fontsize=8)
plt.savefig('output/graph/grafico_2_balanza_comercial_servicios.png', bbox_inches='tight', dpi=200)
plt.tight_layout()
fig.text(0.5, -0.02, 'Elaboracion propia con datos del BCRP.', fontfamily='serif', fontsize=7.5, ha='center', va='top', fontstyle='italic')
plt.show()


In [ ]:
# Datos de Ingreso Primario y Secundario
plot_table(df[['CC_Ingreso_Primario_Pub', 'CC_Ingreso_Primario_Priv', 'CC_Ingreso_Primario', 'CC_Remesas', 'CC_Ingreso_Secundario']].round(1).reset_index(), title='Tabla 4: Ingreso Primario y Secundario (millones US$)', note='Elaboracion propia con datos del BCRP.', fname='output/tables/tabla_4_ingreso_primario_secundario.png')


In [ ]:
# Ingreso Primario e Ingreso Secundario
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
ax1.plot(df.index, df['CC_Ingreso_Primario_Pub'], 'o-', color='black', markerfacecolor='white', markersize=4, label='Sector Publico')
ax1.plot(df.index, df['CC_Ingreso_Primario_Priv'], 's--', color='black', markerfacecolor='black', markersize=4, label='Sector Privado')
ax1.plot(df.index, df['CC_Ingreso_Primario'], '^-', color='black', markerfacecolor='white', markersize=4, label='Total')
ax1.axhline(0, color='black', linewidth=0.3)
ax1.set_title('Ingreso Primario', fontsize=10)
ax1.set_ylabel('Millones de US$')
ax1.set_xlabel('Año')
ax1.legend(frameon=False, fontsize=8)
ax2.plot(df.index, df['CC_Ingreso_Secundario'], 'o-', color='black', markerfacecolor='white', markersize=5, label='Ing. Secundario total')
ax2.plot(df.index, df['CC_Remesas'], 's--', color='black', markerfacecolor='black', markersize=4, label='del cual: Remesas')
ax2.axhline(0, color='black', linewidth=0.3)
ax2.set_title('Ingreso Secundario y Remesas', fontsize=10)
ax2.set_ylabel('Millones de US$')
ax2.set_xlabel('Año')
ax2.legend(frameon=False, fontsize=8)
plt.savefig('output/graph/grafico_3_ingreso_primario_secundario.png', bbox_inches='tight', dpi=200)
plt.tight_layout()
fig.text(0.5, -0.02, 'Elaboracion propia con datos del BCRP.', fontfamily='serif', fontsize=7.5, ha='center', va='top', fontstyle='italic')
plt.show()


In [ ]:
# Datos de Cuenta Financiera
plot_table(df[['CF_Sector_Privado', 'CF_SectorPublico', 'CF_CortoPlazo', 'Cuenta_Financiera']].round(1).reset_index(), title='Tabla 5: Componentes de Cuenta Financiera (millones US$)', note='Elaboracion propia con datos del BCRP.', fname='output/tables/tabla_5_componentes_cf.png')


In [ ]:
# Cuenta Financiera y sus componentes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
x = range(len(df))
ax1.bar(x, df['CF_Sector_Privado'], color='white', edgecolor='black', linewidth=0.5, label='Sector Privado')
ax1.bar(x, df['CF_SectorPublico'], bottom=df['CF_Sector_Privado'], color='darkgray', edgecolor='black', linewidth=0.5, label='Sector Publico')
ax1.bar(x, df['CF_CortoPlazo'], bottom=df['CF_Sector_Privado']+df['CF_SectorPublico'], color='lightgray', edgecolor='black', linewidth=0.5, label='Corto Plazo')
ax1.set_xticks(list(x))
ax1.set_xticklabels(df.index)
ax1.set_ylabel('Millones de US$')
ax1.set_title('Componentes de Cuenta Financiera', fontsize=10)
ax1.legend(frameon=False, fontsize=7)
ax1.axhline(0, color='black', linewidth=0.3)
ax2.plot(df.index, df['Cuenta_Financiera'], 'o-', color='black', markerfacecolor='white', markeredgecolor='black', markersize=5, label='Cuenta Financiera (neta)')
ax2.axhline(0, color='black', linestyle=':', linewidth=0.5)
ax2.set_ylabel('Millones de US$')
ax2.set_xlabel('Año')
ax2.set_title('Cuenta Financiera (neta)', fontsize=10)
ax2.legend(frameon=False)
plt.savefig('output/graph/grafico_4_cf_componentes.png', bbox_inches='tight', dpi=200)
plt.tight_layout()
fig.text(0.5, -0.02, 'Elaboracion propia con datos del BCRP.', fontfamily='serif', fontsize=7.5, ha='center', va='top', fontstyle='italic')
plt.show()


In [ ]:
# Datos de Activos y Pasivos
plot_table(df[['CF_SectorPrivado_Activos', 'CF_SectorPrivado_Pasivos', 'CF_SectorPublico_Activos', 'CF_SectorPublico_Pasivos', 'CF_CortoPlazo_Activos', 'CF_CortoPlazo_Pasivos']].round(1).reset_index(), title='Tabla 6: Activos y Pasivos por Sector (millones US$)', note='Elaboracion propia con datos del BCRP.', fname='output/tables/tabla_6_activos_pasivos_sector.png')


In [ ]:
# Desglose de Sectores: Activos y Pasivos
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
sectores = [
    ('Sector Privado', 'CF_SectorPrivado_Activos', 'CF_SectorPrivado_Pasivos'),
    ('Sector Publico', 'CF_SectorPublico_Activos', 'CF_SectorPublico_Pasivos'),
    ('Corto Plazo', 'CF_CortoPlazo_Activos', 'CF_CortoPlazo_Pasivos'),
]
for ax, (nombre, activos, pasivos) in zip(axes, sectores):
    ax.plot(df.index, df[activos], 'o-', color='black', markerfacecolor='white', markersize=4, label='Activos')
    ax.plot(df.index, df[pasivos], 's--', color='black', markerfacecolor='black', markersize=4, label='Pasivos')
    ax.axhline(0, color='black', linewidth=0.3)
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel('Año')
    ax.legend(frameon=False, fontsize=7)
axes[0].set_ylabel('Millones de US$')
plt.savefig('output/graph/grafico_5_activos_pasivos_sector.png', bbox_inches='tight', dpi=200)
plt.tight_layout()
fig.text(0.5, -0.02, 'Elaboracion propia con datos del BCRP.', fontfamily='serif', fontsize=7.5, ha='center', va='top', fontstyle='italic')
plt.show()


In [ ]:
# Resultado de BP y Variacion de RIN
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(df.index, df['Resultado_BP'], 'o-', color='black', markerfacecolor='white', markeredgecolor='black', markersize=5, label='Resultado de Balanza de Pagos')
ax.plot(df.index, df['Efecto_Valuacion'], 's--', color='black', markerfacecolor='white', markersize=4, label='Efecto Valuacion')
ax.plot(df.index, df['Var_Saldo_RIN'], 'D-', color='black', markerfacecolor='black', markersize=4, label='Variacion del Saldo de RIN')
ax.axhline(0, color='black', linewidth=0.4)
ax.set_ylabel('Millones de US$')
ax.set_xlabel('Año')
ax.legend(frameon=False, fontsize=8)
plt.savefig('output/graph/grafico_6_resultado_bp_rin.png', bbox_inches='tight', dpi=200)
plt.tight_layout()
fig.text(0.5, -0.02, 'Elaboracion propia con datos del BCRP.', fontfamily='serif', fontsize=7.5, ha='center', va='top', fontstyle='italic')
plt.show()


In [ ]:
# Datos de Resultado y RIN
plot_table(df[['Resultado_BP', 'Efecto_Valuacion', 'Var_Saldo_RIN']].round(1).reset_index(), title='Tabla 7: Resultado de BP y Variacion de RIN (millones US$)', note='Elaboracion propia con datos del BCRP.', fname='output/tables/tabla_7_resultado_bp_rin.png')
